<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/04_train_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Mounted at /content/drive
Thiết bị tính toán đang dùng: cuda


In [2]:
# ===== CELL 2: PYTORCH DATASET CHO PINN =====
from torch.utils.data import Dataset, DataLoader # Import hai lớp quản lý Dataset và nạp dữ liệu theo Batch
import numpy as np                               # Import NumPy để nạp các file lưu dưới dạng mảng .npy

class PMSMPINNDataset(Dataset):
    """
    Lớp Custom Dataset dùng để nạp Input (X), Target (Y) và Chuỗi thời gian (Time)
    phục vụ cho tính toán vi phân trong hàm Loss Vật Lý của PINN.
    """
    def __init__(self, data_dir, split='train'):
        # Nạp mảng Input đã chuẩn hóa: ud, uq, Id, Iq, nm
        self.X = np.load(os.path.join(data_dir, f'X_{split}.npy'))

        # Nạp mảng Target đã chuẩn hóa: Rs, Ld, Lq
        self.Y = np.load(os.path.join(data_dir, f'Y_{split}.npy'))

        # Nạp mảng chuỗi thời gian Time để hỗ trợ tính toán đạo hàm theo thời gian
        self.time = np.load(os.path.join(data_dir, f'Time_{split}.npy'))

        # Chuyển đổi dữ liệu từ dạng NumPy Array sang Tensor của PyTorch (kiểu float32)
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.Y = torch.tensor(self.Y, dtype=torch.float32)
        self.time = torch.tensor(self.time, dtype=torch.float32)

    def __len__(self):
        # Trả về tổng số lượng mẫu dữ liệu có trong tập dữ liệu
        return len(self.X)

    def __getitem__(self, idx):
        # Trả về bộ 3 giá trị tương ứng với chỉ số (index) idx: Input, Target và Time
        return self.X[idx], self.Y[idx], self.time[idx]

# Nạp 3 tập dữ liệu đã phân chia từ thư mục processed trên Drive
train_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='train')       # Tập Train (Data1, Data2, Data3)
val_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='validation') # Tập Validation (Data4)
test_dataset = PMSMPINNDataset(PROCESSED_DATA_DIR, split='test')         # Tập Test (Data5)

# Khởi tạo DataLoader để chia dữ liệu thành từng Batch (kích thước 128 mẫu/batch)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)  # Trộn ngẫu nhiên tập Train khi nạp
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)     # Giữ nguyên thứ tự tập Validation
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)   # Giữ nguyên thứ tự tập Test

print(f'Đã nạp Dataset PINN thành công! Train: {len(train_dataset)} mẫu.')

Đã nạp Dataset PINN thành công! Train: 120000 mẫu.


In [3]:
# ===== CELL 3: KIẾN TRÚC MẠNG PINN & PHYSICS LOSS CHUẨN HÓA =====
import torch.nn as nn
import joblib
import numpy as np
import torch
import os

# Nạp bộ Scaler đã fit ở bước Preprocessing mới
X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32).to(device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32).to(device)
y_mean = torch.tensor(Y_scaler.mean_, dtype=torch.float32).to(device)
y_scale = torch.tensor(Y_scaler.scale_, dtype=torch.float32).to(device)

class PMSM_PINN(nn.Module):
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_PINN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

def compute_physics_loss(inputs_scaled, outputs_scaled, pole_pairs=4, psi_m=0.175):
    """
    Physics Loss chuẩn hóa theo công suất sụt áp Stator
    """
    inputs_real = inputs_scaled * x_scale + x_mean
    outputs_real = outputs_scaled * y_scale + y_mean

    ud = inputs_real[:, 0]
    uq = inputs_real[:, 1]
    Id = inputs_real[:, 2]
    Iq = inputs_real[:, 3]
    nm = inputs_real[:, 4]

    Rs_est = outputs_real[:, 0]
    Ld_est = outputs_real[:, 1]
    Lq_est = outputs_real[:, 2]

    omega_m = nm * (2.0 * np.pi / 60.0)
    omega_e = pole_pairs * omega_m

    # Phương trình điện áp d-q tiêu chuẩn
    ud_calc = Rs_est * Id - omega_e * Lq_est * Iq
    uq_calc = Rs_est * Iq + omega_e * (Ld_est * Id + psi_m)

    loss_ud = torch.mean((ud - ud_calc) ** 2)
    loss_uq = torch.mean((uq - uq_calc) ** 2)

    return loss_ud + loss_uq


# Khởi tạo mô hình PINN và chuyển lên thiết bị tính toán
model_pinn = PMSM_PINN().to(device)
print(model_pinn) # In cấu trúc mạng Neural ra màn hình

PMSM_PINN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [5]:
# ===== CELL 4: HUẤN LUYỆN PINN (ĐÃ THÊM CHÚ THÍCH CƠ SỞ LÝ THUYẾT) =====
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# ------------------------------------------------------------------------------------------------
# 1. CƠ SỞ LÝ THUYẾT CẤU TRÚC HÀM MẤT MÁT HỢP NHẤT (UNIFIED LOSS FUNCTION)
# - Công thức tổng quát: L_total = L_data + lambda_p * L_physics[cite: 5].
# - Data Loss (L_data): Đo lường sai lệch giữa tham số dự đoán với nhãn thực nghiệm (MSE).
# - Physics Loss (L_physics): Ép mạng tuân thủ định luật Euler-Lagrangian/Maxwell của PMSM.
# - Trọng số cân bằng lambda_p = 1e-4: Cân bằng quy mô giữa L_data (chuẩn hóa [0,1]) và L_physics (Volt)[cite: 5].
# - Nguồn tham khảo: Zhao et al., 2022 (IEEE Transactions on Power Electronics).
# ------------------------------------------------------------------------------------------------
criterion_data = nn.MSELoss() # Hàm tính Data Loss bằng MSE
lambda_p = 1e-4               # Trọng số Physics Loss cân bằng biên độ
epochs = 100                  # Tổng số lượt lặp huấn luyện toàn bộ dữ liệu (150 Epochs)
best_val_loss = float('inf')  # Khởi tạo giá trị Validation Loss tốt nhất
checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth')

# ------------------------------------------------------------------------------------------------
# 2. CƠ SỞ LÝ THUYẾT VỀ THUẬT TOÁN TỐI ƯU HÓA & NẮC ĐIỀU CHỈNH TỐC ĐỘ HỌC (SCHEDULER)
# - Optimizer Adam: Thuật toán bậc nhất điều chỉnh tốc độ học thích ứng dựa trên Moment bậc 1 và 2.
# - CosineAnnealingLR: Hạ dần tốc độ học (Learning Rate) từ 0.001 -> 0.00001 theo nhịp đường cong Cosine.
#   Ở những epoch cuối, lr cực mịn giúp Adam nhẫn nại tinh chỉnh sụt áp thuần trở Rs*Iq xuống mức sai số tối ưu.
# - Gradient Clipping (clip_grad_norm_): Chống hiện tượng Bùng nổ Gradient (Exploding Gradient).
# ------------------------------------------------------------------------------------------------
optimizer = optim.Adam(model_pinn.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

print("=== BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH PINN ===")

for epoch in range(1, epochs + 1):
    # --- GIAI ĐOẠN 1: TẬP TRUNG HUẤN LUYỆN (TRAINING PHASE) ---
    model_pinn.train() # Chuyển mô hình sang chế độ huấn luyện
    train_data_loss = 0.0
    train_phys_loss = 0.0

    # Lặp qua từng Batch dữ liệu (Batch size = 128)
    for inputs, targets, _ in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        # Đặt lại Gradient về 0 trước khi lan truyền ngược bước mới
        optimizer.zero_grad()

        # Lan truyền tiến (Forward Pass): Dự đoán đầu ra
        outputs = model_pinn(inputs)

        # 1. Tính Mất mát Dữ liệu (Data Loss)
        loss_d = criterion_data(outputs, targets)

        # 2. Tính Mất mát Luật Vật lý (Physics Loss)
        loss_p = compute_physics_loss(inputs, outputs)

        # 3. Tính TỔNG HÀM LOSS HỢP NHẤT (L_total = L_data + lambda_p * L_physics)
        loss_total = loss_d + lambda_p * loss_p

        # Lan truyền ngược (Backward Pass): Tính Gradient qua cơ chế Autograd theo Quy tắc Chuỗi (Chain Rule)
        loss_total.backward()

        # Kỹ thuật Gradient Clipping: Giới hạn chuẩn gradient tối đa = 1.0 để chống nổ gradient
        torch.nn.utils.clip_grad_norm_(model_pinn.parameters(), max_norm=1.0)

        # Cập nhật trọng số mạng Neural bằng thuật toán Adam
        optimizer.step()

        # Cộng dồn giá trị Loss để thống kê theo kích thước Batch
        train_data_loss += loss_d.item() * inputs.size(0)
        train_phys_loss += loss_p.item() * inputs.size(0)

    # Cập nhật hạ tốc độ học theo Cosine Scheduler sau mỗi Epoch
    scheduler.step()

    # Tính trung bình các loại Loss trên toàn bộ tập dữ liệu Train
    train_data_loss /= len(train_dataset)
    train_phys_loss /= len(train_dataset)

    # --- GIAI ĐOẠN 2: GIÁM SÁT ĐÁNH GIÁ (VALIDATION PHASE) ---
    model_pinn.eval() # Bật chế độ đánh giá
    val_loss = 0.0

    with torch.no_grad(): # Tắt tính toán Gradient để tiết kiệm bộ nhớ GPU và tăng tốc
        for inputs, targets, _ in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model_pinn(inputs)
            loss_d = criterion_data(outputs, targets) # Tính sai số trên tập Validation
            val_loss += loss_d.item() * inputs.size(0)

    val_loss /= len(val_dataset) # Tính trung bình Validation Loss

    # In thông tin tiến độ huấn luyện sau mỗi 15 Epochs
    if epoch % 10 == 0 or epoch == 1:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch:3d}/{epochs}] | lr: {current_lr:.6f} | Data Loss: {train_data_loss:.6f} | Phys Loss: {train_phys_loss:.4f} | Val Loss: {val_loss:.6f}")

    # Tự động lưu Checkpoint chứa bộ trọng số mô hình tốt nhất vào Google Drive
    if val_loss < best_val_loss:
        best_val_loss = val_loss # Cập nhật kỷ lục Validation Loss nhỏ nhất
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pinn.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, checkpoint_path)

print(f"\n--> HOÀN THÀNH HUẤN LUYỆN PINN! Weights tốt nhất lưu tại: {checkpoint_path}")

=== BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH PINN ===
Epoch [  1/100] | lr: 0.001000 | Data Loss: 0.264275 | Phys Loss: 4914.1230 | Val Loss: 0.317996
Epoch [ 10/100] | lr: 0.000976 | Data Loss: 0.220305 | Phys Loss: 4914.1444 | Val Loss: 0.298630
Epoch [ 20/100] | lr: 0.000905 | Data Loss: 0.186749 | Phys Loss: 4914.1562 | Val Loss: 0.332459
Epoch [ 30/100] | lr: 0.000796 | Data Loss: 0.172713 | Phys Loss: 4914.2065 | Val Loss: 0.335790
Epoch [ 40/100] | lr: 0.000658 | Data Loss: 0.164099 | Phys Loss: 4914.1676 | Val Loss: 0.347185
Epoch [ 50/100] | lr: 0.000505 | Data Loss: 0.155326 | Phys Loss: 4914.1803 | Val Loss: 0.343685
Epoch [ 60/100] | lr: 0.000352 | Data Loss: 0.147867 | Phys Loss: 4914.1500 | Val Loss: 0.346241
Epoch [ 70/100] | lr: 0.000214 | Data Loss: 0.140659 | Phys Loss: 4914.1824 | Val Loss: 0.319132
Epoch [ 80/100] | lr: 0.000105 | Data Loss: 0.135656 | Phys Loss: 4914.1784 | Val Loss: 0.333476
Epoch [ 90/100] | lr: 0.000034 | Data Loss: 0.132695 | Phys Loss: 4914.1822 | Val Loss:

In [6]:
# ===== CELL 5: ĐÁNH GIÁ PINN TRÊN TẬP TEST =====

# Nạp bản trọng số tốt nhất đã lưu trên Google Drive ra để đánh giá
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth'))
model_pinn.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trạng thái trọng số
model_pinn.eval() # Chuyển mô hình sang chế độ dự đoán/đánh giá

all_preds = []   # Mảng chứa toàn bộ kết quả dự đoán từ mạng PINN
all_targets = [] # Mảng chứa toàn bộ nhãn thực tế từ mô phỏng

with torch.no_grad(): # Tắt tính toán gradient khi kiểm tra tập Test
    for inputs, targets, _ in test_loader:
        inputs = inputs.to(device)                     # Đẩy dữ liệu lên GPU
        outputs = model_pinn(inputs)                  # Mạng PINN dự đoán
        all_preds.append(outputs.cpu().numpy())       # Lưu dự đoán về CPU dưới dạng NumPy
        all_targets.append(targets.numpy())           # Lưu nhãn thực tế dưới dạng NumPy

# Ghép các Batch nhỏ thành mảng 2D hoàn chỉnh
all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# Giải chuẩn hóa (Inverse Transform) kết quả dự đoán và nhãn thực tế về đơn vị thực (Ohm và Henry)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số: Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình (PINN): {mape[0]:.2f}%") # In phần trăm sai số của Rs
print(f"Sai số Ld trung bình (PINN): {mape[1]:.2f}%") # In phần trăm sai số của Ld
print(f"Sai số Lq trung bình (PINN): {mape[2]:.2f}%") # In phần trăm sai số của Lq

=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===
Sai số Rs trung bình (PINN): 6.86%
Sai số Ld trung bình (PINN): 0.07%
Sai số Lq trung bình (PINN): 0.04%
